# PLN Sesión 1: La mirada de la máquina (NER y Escala)

---

## ¿Por qué?

En las sesiones de expresiones regulares se aprendió a buscar patrones basados en la **forma gráfica**: "si termina en -ing", "si es una sigla en mayúsculas", "si está entre comillas". Es una técnica quirúrgica y potente, pero tiene un límite estructural: no sabe qué es lo que ha encontrado.

Regex no puede distinguir si *Apple* es una fruta o una multinacional, ni puede saber que *Siri* es una entidad si no se escribe una regla específica para ella. El **Procesamiento del Lenguaje Natural (PLN)** moderno no se basa en reglas, sino en **modelos estadísticos** que han "leído" millones de textos para aprender a predecir el contexto. 

Esta sesión trata sobre el salto de la regla estadística a la intuición del modelo.

## ¿Qué?

El **Reconocimiento de Entidades Nombradas (NER)** es la capacidad de un modelo de identificar y clasificar automáticamente unidades lingüísticas que representan objetos del mundo real: personas, organizaciones, lugares, fechas o productos. 

No se realiza mediante búsquedas en una lista (diccionario), sino analizando el entorno de la palabra. 

## ¿Para qué?

Para el profesional de la lengua, el NER permite:
- **Mapear actores:** Extraer quién es quién en un corpus de miles de noticias sin necesidad de una lectura manual exhaustiva.
- **Anonimizar:** Detectar datos sensibles para proteger la privacidad.
- **Auditar modelos:** Entender dónde falla la tecnología y por qué (sesgos, neologismos, errores de contexto).

## ¿Cómo?

Se utilizará **spaCy**, la biblioteca estándar en la industria, para procesar desde frases aisladas hasta noticias completas, pasando de la "magia" inicial a la auditoría profesional.

---
## 1. Preparación del entorno

A diferencia de las bibliotecas estándar de Python, los sistemas de PLN requieren **modelos de lengua** pre-entrenados. Estos modelos contienen el conocimiento estadístico necesario para analizar el texto. En cada nueva sesión de trabajo (como en Google Colab), es imperativo descargar y cargar el modelo correspondiente.

In [ ]:
!python -m spacy download es_core_news_sm --quiet
import spacy
from spacy import displacy
import pandas as pd

nlp = spacy.load("es_core_news_sm")
print("Modelo de lengua española cargado.")

---
## 2. La "magia" del NER

Considérese la siguiente frase. Contiene siglas, nombres de empresas, nombres de personas y productos. Con Regex, se requeriría una docena de reglas complejas. Con spaCy, solo es necesario procesar el texto con el modelo.

In [ ]:
texto_complejo = """
El presidente de la RAE se reunió en la sede de la Fundéu con representantes de Apple y Microsoft 
para debatir cómo Siri y ChatGPT están transformando el español en San Francisco y Madrid.
"""

doc = nlp(texto_complejo)

# Visualización inmediata
displacy.render(doc, style="ent", jupyter=True)

### Reflexión lingüística
- ¿Cómo identifica el modelo que *Siri* es una entidad si no es una palabra común en español?
- ¿Por qué se ha clasificado a la *RAE* como organización y a *San Francisco* como lugar?
- **El modelo no posee una lista de nombres; desarrolla una "intuición" estadística analizando los tokens vecinos y la estructura sintáctica de la oración (lo que técnicamente se conoce como tensores de contexto).**

---
## 3. La auditoría lingüística: ¿Dónde falla el modelo?

La tecnología no es infalible. Como profesionales de la lengua, la labor consiste no solo en emplear la herramienta, sino en **auditar sus resultados**. 

A continuación, se presentan ejemplos que pueden inducir a error al modelo (polisemia, mayúsculas engañosas, neologismos):

In [ ]:
ejemplos_criticos = [
    "Mañana viajaré a León.",
    "El León es el rey de la selva.",
    "He comprado una manzana en Apple Store.",
    "La Rosa de Jericó es una planta, pero Rosa se fue a Sevilla.",
    "El sistema de phishing detectó un scroll infinito en el feed."
]

for texto in ejemplos_criticos:
    doc_critico = nlp(texto)
    print(f"--- Analizando: {texto}")
    displacy.render(doc_critico, style="ent", jupyter=True)

**Pregunta para el debate:**
¿En qué casos se han producido errores? ¿Se trata de un problema de mayúsculas, de significado (polisemia) o de falta de vocabulario técnico?

---
## 4. Reto Final: El mapa de una noticia real

En este punto el PLN demuestra su valor profesional: la **escala**. Se procesará una noticia completa para extraer automáticamente todos sus protagonistas.

### Procedimiento:
1. Seleccionar una noticia reciente en un diario digital (El País, El Mundo, ABC, etc.).
2. Copiar el cuerpo de la noticia y pegarlo en la variable `noticia_real`.
3. Ejecutar la celda para generar el mapa de entidades.

In [ ]:
noticia_real = """
PEGA AQUÍ TU NOTICIA
"""

if noticia_real.strip() == "PEGA AQUÍ TU NOTICIA":
    print("Por favor, pega una noticia para continuar.")
else:
    doc_noticia = nlp(noticia_real)
    
    # Extracción de entidades
    entidades = [
        {"Texto": ent.text, "Etiqueta": ent.label_, "Descripción": spacy.explain(ent.label_)}
        for ent in doc_noticia.ents
    ]
    
    df = pd.DataFrame(entidades)
    
    print(f"Se han identificado {len(entidades)} entidades.\n")
    
    # Frecuencia de entidades
    if not df.empty:
        print("--- Top 10 entidades más frecuentes ---")
        print(df['Texto'].value_counts().head(10))
        
        print("\n--- Visualización completa ---")
        displacy.render(doc_noticia, style="ent", jupyter=True)
    else:
        print("No se han detectado entidades en el texto proporcionado.")

---
## Conclusiones

### Nota técnica: El fundamento estadístico y sus límites

El Reconocimiento de Entidades (NER) no se basa en una comprensión conceptual del mundo, sino en la **probabilidad estadística**. El modelo `es_core_news_sm` ha sido entrenado procesando millones de palabras procedentes, principalmente, de fuentes periodísticas y literarias estandarizadas. 

**Implicaciones del entrenamiento:**
- **Sesgo hacia el estándar:** El modelo identifica con alta precisión entidades recurrentes en el lenguaje general (instituciones, topónimos clásicos, personajes públicos).
- **Puntos ciegos técnicos:** Ante neologismos o términos de nicho (ej. *phishing*, *mansplaining*), el modelo carece de referencias estadísticas previas. Al no haber sido expuesto a estos términos durante su fase de entrenamiento, no puede asignarles una categoría de entidad fiable.
- **Dependencia del contexto:** La clasificación depende de las palabras que rodean a la entidad. Un cambio en la capitalización o en la estructura de la frase puede alterar drásticamente la decisión del modelo.

**Criterio profesional:**
La eficacia de un sistema de PLN está condicionada por la representatividad de sus datos de entrenamiento. El profesional de la lengua debe evaluar si el modelo generalista es adecuado para el dominio específico de trabajo o si se requiere un modelo especializado (entrenado con corpus técnicos, médicos o jurídicos).

---

- **Regex** busca formas; **PLN** analiza funciones.
- El **NER** permite el salto del nivel de carácter al nivel de entidades del mundo real.
- La **escala** es el factor diferencial: el PLN destaca cuando el volumen de texto impide una lectura humana inmediata.

**Siguiente fase:** ¿Cómo es posible determinar si dos entidades (como "bulo" y "fake news") comparten significado pese a su distinta forma? Se abordará en la sesión sobre **Semántica y Vectores**.